In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "amount",
    when(col("amount").isNull(), "1000").otherwise(col("amount"))
)

In [0]:
df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("updated_date", col("updated_date").cast("date"))

In [0]:
df = df.withColumn("bonus", col("amount") * 0.1)

df = df.withColumn(
    "category",
    when(col("amount") > 20000, "High").otherwise("Low")
)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def bucket(amount):
    if amount < 10000:
        return "Low"
    elif 10000 <= amount <= 30000:
        return "Medium"
    else:
        return "High"

bucket_udf = udf(bucket, StringType())

df = df.withColumn("amount_bucket", bucket_udf(col("amount")))

In [0]:
df.write.mode("overwrite").saveAsTable("orders_table")

In [0]:
new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
    (9, "C009", "Speaker", "12000", "2024-01-08")
]

new_df = spark.createDataFrame(new_data, columns)

In [0]:
new_df = new_df.withColumn(
    "amount",
    when(col("amount").isNull(), "1000").otherwise(col("amount"))
)

new_df = new_df.withColumn("amount", col("amount").cast("int")) \
               .withColumn("updated_date", col("updated_date").cast("date"))

new_df = new_df.withColumn("bonus", col("amount") * 0.1)

new_df = new_df.withColumn(
    "category",
    when(col("amount") > 20000, "High").otherwise("Low")
)

new_df = new_df.withColumn("amount_bucket", bucket_udf(col("amount")))

In [0]:
existing_df = spark.table("orders_table")

In [0]:
from pyspark.sql.functions import col, lit

new_df = new_df.filter(col("updated_date") > lit("2024-01-05"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

combined_df = existing_df.union(new_df)

final_df = combined_df.withColumn("rn", row_number().over(window)) \
                      .filter(col("rn") == 1) \
                      .drop("rn")

In [0]:
final_df.write.mode("overwrite").saveAsTable("orders_table")

In [0]:
df.show()

+--------+-----------+----------+------+------------+------+--------+-------------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|amount_bucket|
+--------+-----------+----------+------+------------+------+--------+-------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|         High|
|       2|       C002|    Mobile|  1000|  2024-01-02| 100.0|     Low|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|     Low|       Medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|         High|
|       5|       C005|Headphones|  1000|  2024-01-05| 100.0|     Low|          Low|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|       Medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     Low|       Medium|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     Low|          Low|
+--------+-----------+----------+------+------------+------+--------+-------